# MLOps - Caso de uso 2: Reingreso hospitalario en pacientes diabéticos

> **ADVERTENCIA CLÍNICA.** Este notebook es **experimental y con fines académicos**. Sus resultados **no sustituyen decisiones médicas** ni deben usarse para el cuidado de pacientes. Las salidas deben contrastarse con conocimiento experto.

**Problema:** predecir si un paciente diabético será reingresado en **menos de 30 días** (`readmitted == "<30"` → 1; `">30"` o `"NO"` → 0).
**Datos:** *Diabetes 130-US hospitals for years 1999-2008* (UCI, id 296): ~101.766 encuentros, 50 columnas, ~11 % de positivos.

**Por qué no basta la accuracy:** con ~11 % de positivos, un modelo que predice "nadie reingresa" obtiene ~89 % de accuracy sin detectar a ningún paciente. Se reportan ROC-AUC, average precision (PR-AUC), precision, recall y F1 de la clase positiva, y se registra también `baseline_accuracy`.

Tags usados: `parameters`, `mlops:config`, `mlops:data`, `mlops:preprocessing`, `mlops:training`, `mlops:evaluation`, `mlops:export`.

## Parámetros
Papermill sobrescribe esta celda. Además de las 4 variables del contrato se declaran `GOLD_DB_PATH`, `GOLD_TABLE` y `DATA_PATH`.

In [ ]:
# Papermill injected parameters (do not edit this cell manually)
# These are overwritten at runtime by the pipeline.
MODEL_OUTPUT_PATH = "./model.joblib"
PIPELINE_ID = "local-dev"
MLFLOW_TRACKING_URI = "http://localhost:5000"
MLFLOW_RUN_ID = "local-dev"
# Parámetros propios del caso de uso (valores por defecto)
GOLD_DB_PATH = ""
GOLD_TABLE = "gold_diabetes"
DATA_PATH = "./diabetic_data.csv"

## Configuración
Nombre y versión con los que la plataforma registra el modelo.

In [ ]:
# ============================================================
# mlops:config - Metadatos del modelo
# ============================================================
MODEL_NAME = "diabetes-readmission-classifier"
VERSION = "1"

print(f"Modelo: {MODEL_NAME} v{VERSION}")
print(f"Pipeline ID: {PIPELINE_ID}")

## Datos
Prioridad de lectura: (a) capa **gold** de la plataforma (DuckDB, solo lectura) si `GOLD_DB_PATH` está definido; (b) CSV crudo de UCI en `DATA_PATH`. El CSV se lee sin convertir centinelas, para que la normalización quede documentada en el siguiente paso.

In [ ]:
# ============================================================
# mlops:data - Carga de datos (gold -> CSV crudo de UCI)
# ============================================================
import os
import numpy as np
import pandas as pd

# Las rutas pueden llegar por parámetros de Papermill o por variables de entorno.
if not GOLD_DB_PATH and os.environ.get("GOLD_DB_PATH"):
    GOLD_DB_PATH = os.environ["GOLD_DB_PATH"]
    GOLD_TABLE = os.environ.get("GOLD_TABLE", GOLD_TABLE)

df = None
MODO_DATOS = None

# (a) Capa gold de la plataforma (ya normalizada), DuckDB en modo solo lectura
if GOLD_DB_PATH:
    try:
        # TODO: ajustar a la convención real de lectura de gold de la plataforma
        import duckdb
        con = duckdb.connect(GOLD_DB_PATH, read_only=True)
        try:
            df = con.execute(f'SELECT * FROM "{GOLD_TABLE}"').df()
        finally:
            con.close()
        MODO_DATOS = "gold"
        FUENTE_DATOS = f"gold DuckDB: {GOLD_DB_PATH} :: {GOLD_TABLE}"
    except Exception as exc:
        print(f"No se pudo leer gold ({exc}); se intenta el CSV crudo.")
        df = None

# (b) CSV crudo de UCI (se conservan los centinelas "?" y "None" tal como vienen)
if df is None:
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(
            f"No se encontró el CSV en '{DATA_PATH}'. Descarga 'diabetic_data.csv' del dataset "
            "'Diabetes 130-US hospitals' (UCI Machine Learning Repository, id 296) y colócalo junto "
            "al notebook (en Colab: súbelo desde el panel de archivos), o define DATA_PATH."
        )
    df = pd.read_csv(DATA_PATH, keep_default_na=False, na_values=[], low_memory=False)
    MODO_DATOS = "crudo"
    FUENTE_DATOS = f"CSV crudo: {DATA_PATH}"

print(f"Modo de datos: {MODO_DATOS} | Fuente: {FUENTE_DATOS}")
print(f"Shape: {df.shape[0]:,} filas x {df.shape[1]} columnas")

print("\nTipos de dato (conteo):")
print(df.dtypes.astype(str).value_counts().to_string())

nulos = (df.isna().mean() * 100).round(2)
print("\n% de nulos reales (NaN) - columnas con nulos:")
print(nulos[nulos > 0].sort_values(ascending=False).head(15).to_string() if (nulos > 0).any() else "Ninguna")

sent = {c: float((df[c] == "?").mean() * 100) for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])}
sent = pd.Series({c: round(v, 2) for c, v in sent.items() if v > 0}).sort_values(ascending=False)
print('\n% de centinelas "?" por columna (el CSV crudo los trae como texto):')
print(sent.head(10).to_string() if len(sent) else "Ninguno")

print("\nDistribución del target:")
if "readmitted" in df.columns:
    print(df["readmitted"].value_counts().to_string())
    print(f'Positivos (readmitted == "<30"): {(df["readmitted"] == "<30").mean() * 100:.2f}%')
elif "target" in df.columns:
    print(df["target"].value_counts().to_string())
    print(f'Positivos: {pd.to_numeric(df["target"]).mean() * 100:.2f}%')
else:
    print("ADVERTENCIA: no se encontró 'readmitted' ni 'target'.")

## Preprocesamiento y gobierno de datos
Normalización documentada (cada paso informa si se aplicó u omitió, así funciona igual con datos crudos o ya normalizados por gold):
centinela `"?"` → NaN; eliminación de identificadores, columnas con >90 % de nulos y constantes; códigos ICD-9 → grupos clínicos; ids de admisión/alta → grupos; edad → punto medio; exclusión de altas por fallecimiento u hospicio (no pueden reingresar).

**Decisiones de gobierno:** `race` se excluye del modelo (se audita igualmente en la evaluación); el split es **por paciente** (`GroupShuffleSplit`) para que un mismo paciente no aparezca en train y test; los transformadores se ajustan solo con train.

In [ ]:
# ============================================================
# mlops:preprocessing - Normalización, split por paciente y transformadores
# ============================================================
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PASOS_APLICADOS, PASOS_OMITIDOS = [], []

def _paso(nombre, aplicado, detalle=""):
    (PASOS_APLICADOS if aplicado else PASOS_OMITIDOS).append(nombre)
    print(f"  [{'APLICADO' if aplicado else 'OMITIDO '}] {nombre}" + (f" -> {detalle}" if detalle else ""))

def _es_num(s):
    return pd.api.types.is_numeric_dtype(s)

def _resumen(d):
    cat = [c for c in d.columns if not _es_num(d[c])]
    return {"filas": len(d), "columnas": d.shape[1], "celdas_nulas": int(d.isna().sum().sum()),
            "cols_categoricas": len(cat), "niveles_categoricos": int(sum(d[c].nunique() for c in cat))}

# Constantes de gobierno de datos
EXCLUIR_DEL_MODELO = ["race"]  # decisión de gobierno: variable sensible fuera del modelo (se audita igualmente)
GRUPOS_ICD = {"Circulatorio", "Respiratorio", "Digestivo", "Diabetes", "Lesiones",
              "Musculoesqueletico", "Genitourinario", "Neoplasias", "Otros", "Desconocido"}
IDS_FALLECIMIENTO_HOSPICIO = {11, 13, 14, 19, 20, 21}   # fallecido o cuidados paliativos

def grupo_icd9(codigo):
    if pd.isna(codigo):
        return "Desconocido"
    c = str(codigo).strip()
    if c[:1] in ("V", "v", "E", "e"):
        return "Otros"
    try:
        v = float(c)
    except ValueError:
        return "Otros"
    ent = int(v)
    if ent == 250:                      return "Diabetes"
    if 390 <= v < 460 or ent == 785:    return "Circulatorio"
    if 460 <= v < 520 or ent == 786:    return "Respiratorio"
    if 520 <= v < 580 or ent == 787:    return "Digestivo"
    if 800 <= v < 1000:                 return "Lesiones"
    if 710 <= v < 740:                  return "Musculoesqueletico"
    if 580 <= v < 630 or ent == 788:    return "Genitourinario"
    if 140 <= v < 240:                  return "Neoplasias"
    return "Otros"

MAPA_TIPO_ADMISION = {1: "Emergencia", 2: "Emergencia", 7: "Emergencia", 3: "Electivo"}
MAPA_ORIGEN_ADMISION = {1: "Referencia", 2: "Referencia", 3: "Referencia", 7: "Urgencias",
                        4: "Traslado", 5: "Traslado", 6: "Traslado", 10: "Traslado",
                        18: "Traslado", 22: "Traslado", 25: "Traslado", 26: "Traslado"}
MAPA_DESTINO_ALTA = {**{i: "Domicilio" for i in (1, 6, 8)},
                     **{i: "Institucion" for i in (2, 3, 4, 5, 9, 10, 12, 15, 16, 17, 22, 23, 24, 27, 28, 29, 30)},
                     7: "Alta voluntaria"}

def _midpoint_edad(txt):
    m = re.match(r"\[(\d+)-(\d+)\)", str(txt))
    return (int(m.group(1)) + int(m.group(2))) / 2 if m else np.nan

print("=== Reporte ANTES de la normalización ===")
antes = _resumen(df)
print(antes)
print("\nPasos de normalización:")

# 1) Centinela "?" -> NaN
n_sent = int(sum((df[c] == "?").sum() for c in df.columns if not _es_num(df[c])))
if n_sent > 0:
    df = df.replace("?", np.nan)
    _paso('Centinela "?" -> NaN', True, f"{n_sent:,} celdas")
else:
    _paso('Centinela "?" -> NaN', False, "no hay centinelas (datos ya normalizados)")

# 2) Target binario
if "readmitted" in df.columns:
    df["target"] = (df["readmitted"].astype(str) == "<30").astype(int)
    df = df.drop(columns=["readmitted"])
    _paso('Target: readmitted == "<30" -> 1; ">30"/"NO" -> 0', True, f"prevalencia={df['target'].mean() * 100:.2f}%")
elif "target" in df.columns:
    df["target"] = pd.to_numeric(df["target"]).astype(int)
    _paso("Target binario", False, "ya existe la columna 'target'")
else:
    raise KeyError("No hay columna 'readmitted' ni 'target'.")

# 3) Exclusión de fallecidos / cuidados paliativos (no pueden reingresar)
if "discharge_disposition_id" in df.columns and _es_num(df["discharge_disposition_id"]):
    n0 = len(df)
    df = df[~df["discharge_disposition_id"].isin(IDS_FALLECIMIENTO_HOSPICIO)].copy()
    _paso("Excluir alta por fallecimiento/hospicio (ids 11,13,14,19,20,21)", True, f"{n0 - len(df):,} filas removidas")
else:
    _paso("Excluir alta por fallecimiento/hospicio", False, "discharge_disposition_id ya agrupado o ausente")

# 4) Columnas sin valor predictivo o con >90% de nulos o constantes
quitar = [c for c in ("encounter_id",) if c in df.columns]
frac_nulos_col = df.isna().mean()
quitar += [c for c in df.columns if frac_nulos_col[c] > 0.90 and c not in quitar and c not in ("target", "patient_nbr")]
quitar += [c for c in df.columns if c not in quitar and c not in ("target", "patient_nbr") and df[c].nunique(dropna=True) <= 1]
if quitar:
    df = df.drop(columns=quitar)
    _paso("Eliminar identificadores, columnas con >90% nulos y constantes", True, ", ".join(quitar))
else:
    _paso("Eliminar identificadores, columnas con >90% nulos y constantes", False, "nada que eliminar")

# 5) Códigos ICD-9 -> categorías clínicas
hechos = []
for col in ("diag_1", "diag_2", "diag_3"):
    if col in df.columns and not set(df[col].dropna().unique()).issubset(GRUPOS_ICD):
        mapa = {c: grupo_icd9(c) for c in df[col].dropna().unique()}
        df[col + "_grupo"] = df[col].map(mapa).fillna("Desconocido")
        df = df.drop(columns=[col])
        hechos.append(col)
_paso("ICD-9 (diag_1/2/3) -> grupos clínicos", bool(hechos), ", ".join(hechos) if hechos else "ya agrupados o ausentes")

# 6) Ids de admisión/alta -> grupos
hechos = []
for col, mapa in (("admission_type_id", MAPA_TIPO_ADMISION), ("admission_source_id", MAPA_ORIGEN_ADMISION),
                  ("discharge_disposition_id", MAPA_DESTINO_ALTA)):
    if col in df.columns and _es_num(df[col]):
        df[col] = df[col].map(mapa).fillna("Otro/Desconocido")
        hechos.append(col)
_paso("Ids de admisión/alta -> grupos", bool(hechos), ", ".join(hechos) if hechos else "ya agrupados o ausentes")

# 7) Edad: intervalo "[70-80)" -> punto medio numérico
if "age" in df.columns and not _es_num(df["age"]):
    df["age"] = df["age"].map(_midpoint_edad)
    _paso("Edad: intervalo -> punto medio numérico", True)
else:
    _paso("Edad: intervalo -> punto medio numérico", False, "ya numérica o ausente")

# Copia para auditoría de equidad (antes de excluir variables del modelo)
AUDITORIA = [c for c in ("gender", "age", "race") if c in df.columns]
meta = df[AUDITORIA].copy()

# 8) Nulos restantes: categóricas -> "Desconocido" (categoría explícita); numéricas se imputan luego (mediana)
cols_cat_tmp = [c for c in df.columns if c not in ("target", "patient_nbr") and not _es_num(df[c])]
nulos_tratados = int(df.drop(columns=["target"]).isna().sum().sum())
NULL_FRAC_TRATADOS = nulos_tratados / float(df.shape[0] * (df.shape[1] - 1))
for c in cols_cat_tmp:
    df[c] = df[c].astype(object).where(df[c].notna(), "Desconocido")
_paso('Nulos categóricos -> "Desconocido"', True, f"{nulos_tratados:,} celdas nulas totales tratadas ({NULL_FRAC_TRATADOS * 100:.2f}% de las celdas)")
_paso("Categorías raras", True, "se agrupan en el OneHotEncoder (min_frequency=0.01), ajustado solo con train")

print("\n=== Reporte DESPUÉS de la normalización ===")
despues = _resumen(df)
print(despues)
print(pd.DataFrame({"antes": antes, "despues": despues}).to_string())
for col in ("diag_1_grupo", "admission_type_id", "discharge_disposition_id"):
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts().head(8).to_string())

# --- Matriz de features, target y grupos (paciente) ---
if "patient_nbr" in df.columns:
    groups = df["patient_nbr"].to_numpy()
else:
    print("ADVERTENCIA: no hay patient_nbr; cada fila se trata como un grupo (riesgo de leakage entre encuentros).")
    groups = np.arange(len(df))
y = df["target"].astype(int).reset_index(drop=True)
X = df.drop(columns=[c for c in ["target", "patient_nbr"] + EXCLUIR_DEL_MODELO if c in df.columns]).reset_index(drop=True)
meta = meta.reset_index(drop=True)
for c in X.columns:
    if not _es_num(X[c]):
        X[c] = X[c].astype(object)
FEATURE_NAMES = list(X.columns)
cols_num = [c for c in FEATURE_NAMES if _es_num(X[c])]
cols_cat = [c for c in FEATURE_NAMES if c not in cols_num]
print(f"\nVariables excluidas del modelo por gobierno de datos: {EXCLUIR_DEL_MODELO}")
print(f"Features del modelo: {len(FEATURE_NAMES)} ({len(cols_num)} numéricas, {len(cols_cat)} categóricas)")

# --- Split por paciente: un paciente NO puede estar en train y test ---
mejor = None
for semilla in range(42, 62):                     # semillas deterministas; se elige la de prevalencia más parecida
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=semilla)
    tr, te = next(gss.split(X, y, groups))
    dif = abs(y.iloc[tr].mean() - y.iloc[te].mean())
    if mejor is None or dif < mejor[0]:
        mejor = (dif, semilla, tr, te)
    if dif < 0.002:
        break
dif, SEMILLA_SPLIT, idx_tr, idx_te = mejor
X_train, X_test = X.iloc[idx_tr].reset_index(drop=True), X.iloc[idx_te].reset_index(drop=True)
y_train, y_test = y.iloc[idx_tr].reset_index(drop=True), y.iloc[idx_te].reset_index(drop=True)
groups_train = groups[idx_tr]
meta_test = meta.iloc[idx_te].reset_index(drop=True)
assert set(groups[idx_tr]).isdisjoint(set(groups[idx_te])), "Hay pacientes en train y test"
print(f"Split por paciente (semilla {SEMILLA_SPLIT}): train={len(X_train):,} test={len(X_test):,} | "
      f"positivos train={y_train.mean() * 100:.2f}% test={y_test.mean() * 100:.2f}% | pacientes compartidos: 0")

# --- Transformadores (se ajustan solo con train, dentro de cada Pipeline/fold) ---
try:
    ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), cols_num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", ohe)]), cols_cat),
    ],
    remainder="drop",
    sparse_threshold=0,
)
print("Preprocesador definido (imputación + escalado numérico; imputación + OneHot categórico).")

## Entrenamiento
Cuatro candidatos (baseline `DummyClassifier`, regresión logística y random forest con pesos de clase, y gradient boosting) dentro de Pipelines con el preprocesamiento. Validación `StratifiedGroupKFold` de 5 folds agrupada por paciente; se elige por **average precision**. Para acotar el tiempo se usa una **submuestra de pacientes** (hasta 40.000 filas) en la comparación y el cálculo del umbral; el ganador se reentrena con todo train.
El **umbral** se fija maximizando F2 (prioriza recall) con predicciones fuera de fold de train, sin usar test.

In [ ]:
# ============================================================
# mlops:training - Candidatos, validación agrupada y umbral
# ============================================================
import time
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, cross_val_predict
from sklearn.metrics import precision_recall_curve

# Submuestreo documentado para mantener el tiempo razonable: la comparación entre modelos y el
# cálculo del umbral usan una submuestra de PACIENTES (nunca se parte a un paciente) de hasta
# CV_MAX_FILAS filas de train. El modelo ganador se reentrena después con TODO train.
CV_MAX_FILAS = 40000
rng = np.random.RandomState(42)
if len(X_train) > CV_MAX_FILAS:
    pacientes = np.unique(groups_train)
    elegidos = pacientes[rng.rand(len(pacientes)) < CV_MAX_FILAS / len(X_train)]
    mask_cv = np.isin(groups_train, elegidos)
else:
    mask_cv = np.ones(len(X_train), dtype=bool)
X_cv, y_cv, g_cv = X_train[mask_cv], y_train[mask_cv], groups_train[mask_cv]
print(f"Validación cruzada sobre {len(X_cv):,} de {len(X_train):,} filas de train "
      f"({y_cv.mean() * 100:.2f}% positivos).")

hgb_kwargs = dict(max_iter=150, learning_rate=0.1, max_depth=6, min_samples_leaf=50, random_state=42)
if "class_weight" in HistGradientBoostingClassifier().get_params():
    hgb_kwargs["class_weight"] = "balanced"
else:
    print("AVISO: esta versión de scikit-learn no soporta class_weight en HistGradientBoosting; se confía en el umbral.")

candidatos = {
    "Dummy (baseline)": DummyClassifier(strategy="prior"),
    "LogisticRegression": LogisticRegression(class_weight="balanced", C=0.1, max_iter=500, random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(**hgb_kwargs),
    "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=20,
                                           class_weight="balanced_subsample", n_jobs=-1, random_state=42),
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)   # agrupado por paciente
scoring = {"roc_auc": "roc_auc", "average_precision": "average_precision"}
filas = []
for nombre, est in candidatos.items():
    t0 = time.time()
    pipe = Pipeline([("prep", clone(preprocessor)), ("model", clone(est))])
    res = cross_validate(pipe, X_cv, y_cv, groups=g_cv, cv=cv, scoring=scoring, n_jobs=1)
    filas.append({"modelo": nombre,
                  "roc_auc_mean": res["test_roc_auc"].mean(), "roc_auc_std": res["test_roc_auc"].std(),
                  "avg_precision_mean": res["test_average_precision"].mean(),
                  "avg_precision_std": res["test_average_precision"].std(),
                  "segundos": time.time() - t0})
    print(f"  {nombre}: AP={filas[-1]['avg_precision_mean']:.4f} AUC={filas[-1]['roc_auc_mean']:.4f} ({filas[-1]['segundos']:.0f}s)")

tabla_cv = pd.DataFrame(filas).sort_values("avg_precision_mean", ascending=False).reset_index(drop=True)
print("\nComparativa (StratifiedGroupKFold, 5 folds, agrupado por paciente):")
print(tabla_cv.round(4).to_string(index=False))

# Criterio de elección: average precision (PR-AUC), más informativa que ROC-AUC con ~11% de positivos
best_name = tabla_cv.loc[0, "modelo"]
cv_average_precision_mean = float(tabla_cv.loc[0, "avg_precision_mean"])
cv_roc_auc_mean = float(tabla_cv.loc[0, "roc_auc_mean"])
print(f"\nMejor modelo: {best_name}")

# Umbral de decisión: maximiza F-beta (beta=2, prioriza recall) con predicciones fuera de fold (OOF)
# sobre train. Se decide SIN mirar test para evitar fuga de información.
pipe_mejor = Pipeline([("prep", clone(preprocessor)), ("model", clone(candidatos[best_name]))])
oof = cross_val_predict(pipe_mejor, X_cv, y_cv, groups=g_cv, cv=cv, method="predict_proba")[:, 1]
prec, rec, thr = precision_recall_curve(y_cv, oof)
BETA_UMBRAL = 2.0   # beta>1 prioriza recall; beta=1 equilibra precision/recall (F1)
beta = BETA_UMBRAL
fbeta = (1 + beta**2) * prec * rec / (beta**2 * prec + rec + 1e-12)
UMBRAL = float(thr[int(np.nanargmax(fbeta[:-1]))])
print(f"Umbral (max F-beta en OOF de train): {UMBRAL:.4f}  "
      f"(F-beta con beta={BETA_UMBRAL}: pondera recall más que precisión, pues perder un reingreso se considera peor que una falsa alarma)")

t0 = time.time()
base_model = Pipeline([("prep", clone(preprocessor)), ("model", clone(candidatos[best_name]))])
base_model.fit(X_train, y_train)
print(f"Modelo final reentrenado con todo train en {time.time() - t0:.0f}s.")

## Evaluación
Métricas sobre test, análisis de equidad por sexo, rango de edad y raza, importancia de variables y registro en MLflow. `accuracy` es obligatoria para la plataforma, pero **debe leerse junto con `baseline_accuracy`, `recall_pos` y `average_precision`**.

**Limitaciones:** datos de 1999-2008 y de 130 hospitales de EE. UU. (posible deriva y sesgo de población); variables sensibles (raza, sexo, edad) que pueden generar disparidades; fuerte desbalance; las importancias son asociaciones, no causas; el umbral prioriza recall y genera muchas falsas alarmas.

In [ ]:
# ============================================================
# mlops:evaluation - Métricas, equidad, importancia y registro en MLflow
# ============================================================
# La plataforma lee la métrica 'accuracy' para decidir el despliegue automático.
import re
import sys
import socket
import subprocess
from urllib.parse import urlparse

import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
from sklearn.inspection import permutation_importance

# --- Métricas globales sobre test ---
proba = base_model.predict_proba(X_test)[:, 1]
y_true = y_test.to_numpy()
y_pred = (proba >= UMBRAL).astype(int)

score = accuracy_score(y_true, y_pred)
baseline_accuracy = float(1.0 - y_true.mean())          # accuracy de "nadie reingresa"
roc_auc = roc_auc_score(y_true, proba)
avg_prec = average_precision_score(y_true, proba)
precision_pos = precision_score(y_true, y_pred, zero_division=0)
recall_pos = recall_score(y_true, y_pred, zero_division=0)
f1_pos = f1_score(y_true, y_pred, zero_division=0)
tasa_alertas = float(y_pred.mean())            # % de pacientes marcados como riesgo

print(f"Umbral aplicado: {UMBRAL:.4f}")
print(f"Accuracy: {score:.4f}  (baseline 'nadie reingresa': {baseline_accuracy:.4f} -> la accuracy sola es engañosa)")
print(f"ROC-AUC: {roc_auc:.4f} | Average precision (PR-AUC): {avg_prec:.4f} | prevalencia test: {y_true.mean():.4f}")
print(f"Clase positiva -> precision: {precision_pos:.4f} | recall: {recall_pos:.4f} | F1: {f1_pos:.4f}")
print(f"Tasa de alertas (pacientes marcados en riesgo): {tasa_alertas * 100:.1f}%")
print("Matriz de confusión (filas=real, columnas=predicho; orden: 0=no reingreso, 1=reingreso<30d):")
print(confusion_matrix(y_true, y_pred))

# --- Equidad por subgrupos ---
def _limpia(txt):
    return re.sub(r"[^A-Za-z0-9_.\- /]", "_", str(txt))

def metricas_subgrupo(mask, min_n=30):
    yt, pr, yp = y_true[mask], proba[mask], y_pred[mask]
    if len(yt) < min_n:
        return None
    out = {"n": int(len(yt)), "prevalencia": float(yt.mean()),
           "recall": float(recall_score(yt, yp, zero_division=0)),
           "precision": float(precision_score(yt, yp, zero_division=0))}
    out["roc_auc"] = float(roc_auc_score(yt, pr)) if 0 < yt.sum() < len(yt) else np.nan
    return out

metricas_sg = {}
if "gender" in meta_test.columns:
    for g in meta_test["gender"].astype(str).unique():
        m = metricas_subgrupo((meta_test["gender"].astype(str) == g).to_numpy())
        if m: metricas_sg[("gender", g)] = m
if "age" in meta_test.columns and _es_num(meta_test["age"]):
    rango = pd.cut(meta_test["age"], bins=[0, 40, 60, 80, 120], labels=["0-40", "40-60", "60-80", "80_o_mas"], right=False)
    for g in rango.dropna().unique():
        m = metricas_subgrupo((rango == g).to_numpy())
        if m: metricas_sg[("age", str(g))] = m
if "race" in meta_test.columns:   # variable excluida del modelo: se audita para detectar sesgo por proxies
    for g in meta_test["race"].astype(str).unique():
        m = metricas_subgrupo((meta_test["race"].astype(str) == g).to_numpy())
        if m: metricas_sg[("race", g)] = m

if metricas_sg:
    tabla_sg = pd.DataFrame(metricas_sg).T
    tabla_sg.index.names = ["variable", "grupo"]
    print("\nAnálisis de equidad por subgrupos (grupos con n >= 30):")
    print(tabla_sg.round(3).to_string())
    brechas = {}
    for var in sorted({k[0] for k in metricas_sg}):
        rec = [v["recall"] for k, v in metricas_sg.items() if k[0] == var and v["n"] >= 100]
        if len(rec) >= 2:
            brechas[var] = max(rec) - min(rec)
    print("Brecha de recall entre subgrupos (n>=100):", {k: round(v, 3) for k, v in brechas.items()})
else:
    brechas = {}
    print("Sin columnas de subgrupo disponibles para el chequeo de equidad.")

# --- Importancia de variables (permutation importance sobre una muestra de test) ---
rng_pi = np.random.RandomState(42)
idx = rng_pi.choice(len(X_test), size=min(3000, len(X_test)), replace=False)
pi = permutation_importance(base_model, X_test.iloc[idx], y_test.iloc[idx], scoring="average_precision",
                            n_repeats=3, random_state=42, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=FEATURE_NAMES).sort_values(ascending=False)
print("\nTop 10 variables por permutation importance (caída de average precision):")
print(imp.head(10).round(4).to_string())
print("\nInterpretación para contrastar con conocimiento clínico:")
print("- Revisar si las variables más influyentes tienen sentido clínico (p. ej. historial de hospitalizaciones")
print("  previas, número de diagnósticos, estancia, destino del alta, edad, control glucémico).")
print("- Importancia alta NO implica causalidad: son asociaciones del dataset 1999-2008.")
print("- Si aparecen variables administrativas dominantes (tipo de pago, especialidad), consultar al experto si son proxies.")

# --- Registro en MLflow ---
# MLFLOW_RUN_ID == "local-dev" significa ejecución manual (fuera de la plataforma). Solo en ese modo
# el notebook se adapta al entorno; bajo el pipeline real cualquier fallo de MLflow se propaga.
LOCAL_DEV = MLFLOW_RUN_ID == "local-dev"
try:
    import mlflow
except ImportError:
    if not LOCAL_DEV:
        raise
    print("mlflow no está instalado; se intenta instalar (solo en modo local)...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mlflow"])
        import mlflow
    except Exception as exc:
        raise ImportError("No se pudo instalar mlflow automáticamente. Ejecuta: pip install mlflow") from exc

def _servidor_accesible(uri, timeout=3):
    if not uri.startswith(("http://", "https://")):
        return True
    p = urlparse(uri)
    puerto = p.port or (443 if p.scheme == "https" else 80)
    try:
        socket.create_connection((p.hostname, puerto), timeout=timeout).close()
        return True
    except OSError:
        return False

tracking_uri = MLFLOW_TRACKING_URI
if LOCAL_DEV and (tracking_uri.startswith("file:") or not _servidor_accesible(tracking_uri)):
    print(f"[modo local] '{MLFLOW_TRACKING_URI}' no es utilizable; se usa sqlite:///mlflow.db")
    tracking_uri = "sqlite:///mlflow.db"
mlflow.set_tracking_uri(tracking_uri)
print(f"MLflow tracking URI en uso: {tracking_uri}")

hiperparametros = {k: v for k, v in base_model.named_steps["model"].get_params().items()
                   if isinstance(v, (int, float, str, bool)) or v is None}

run_ctx = mlflow.start_run(run_id=MLFLOW_RUN_ID) if MLFLOW_RUN_ID != "local-dev" else mlflow.start_run()
with run_ctx:
    mlflow.log_metric("accuracy", score)
    mlflow.log_metric("baseline_accuracy", baseline_accuracy)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("average_precision", avg_prec)
    mlflow.log_metric("recall_pos", recall_pos)
    mlflow.log_metric("precision_pos", precision_pos)
    mlflow.log_metric("f1_pos", f1_pos)
    mlflow.log_metric("umbral", UMBRAL)
    mlflow.log_metric("tasa_alertas", tasa_alertas)
    mlflow.log_metric("cv_average_precision_mean", cv_average_precision_mean)
    mlflow.log_metric("cv_roc_auc_mean", cv_roc_auc_mean)
    for (var, grp), m in metricas_sg.items():
        for nombre in ("recall", "precision", "roc_auc"):
            if not np.isnan(m[nombre]):
                mlflow.log_metric(_limpia(f"sg_{var}_{grp}_{nombre}"), m[nombre])
    for var, b in brechas.items():
        mlflow.log_metric(f"recall_gap_{var}", b)
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("model_version", VERSION)
    mlflow.log_param("best_estimator", best_name)
    mlflow.log_param("data_source", FUENTE_DATOS[:250])
    mlflow.log_param("data_mode", MODO_DATOS)
    mlflow.log_param("n_train", len(X_train))
    mlflow.log_param("n_test", len(X_test))
    mlflow.log_param("null_fraction_treated", round(NULL_FRAC_TRATADOS, 6))
    mlflow.log_param("excluded_features", ",".join(EXCLUIR_DEL_MODELO) or "none")
    mlflow.log_param("normalization_applied", ";".join(PASOS_APLICADOS)[:250])
    mlflow.log_param("normalization_skipped", ";".join(PASOS_OMITIDOS)[:250] or "none")
    for k, v in hiperparametros.items():
        mlflow.log_param(f"hp_{k}", v)
print("Métricas registradas en MLflow.")

## Exportación
Se guarda el modelo completo (con el umbral embebido cuando la versión de scikit-learn lo permite) y se verifica recargándolo. El artefacto espera columnas ya normalizadas.

In [ ]:
# ============================================================
# mlops:export - Guardar el artefacto del modelo
# ============================================================
# Se exporta el modelo COMPLETO (preprocesamiento + clasificador). El umbral de decisión viaja dentro
# del artefacto (FixedThresholdClassifier), de modo que predict() aplica el mismo umbral evaluado.
# Nota: el artefacto espera las columnas YA NORMALIZADAS (esquema silver/gold); la normalización
# de la fase de preprocesamiento (pandas) no forma parte del archivo exportado.
import os
import joblib
import numpy as np

try:
    from sklearn.model_selection import FixedThresholdClassifier
    from sklearn.frozen import FrozenEstimator
    final_model = FixedThresholdClassifier(FrozenEstimator(base_model), threshold=UMBRAL,
                                           response_method="predict_proba", pos_label=1).fit(X_train, y_train)
    UMBRAL_EMBEBIDO = True
except ImportError:
    final_model = base_model
    UMBRAL_EMBEBIDO = False
    print("AVISO: esta versión de scikit-learn no tiene FixedThresholdClassifier/FrozenEstimator (>=1.6).")
    print(f"       Se exporta el pipeline sin umbral: el consumidor debe aplicar predict_proba >= {UMBRAL:.4f}.")

_dir = os.path.dirname(MODEL_OUTPUT_PATH)
if _dir:
    os.makedirs(_dir, exist_ok=True)
joblib.dump(final_model, MODEL_OUTPUT_PATH)

print(f"Modelo guardado en {MODEL_OUTPUT_PATH}")
print(f"Umbral recomendado: {UMBRAL:.4f} ({'embebido en predict()' if UMBRAL_EMBEBIDO else 'NO embebido: aplicar manualmente'})")
print(f"Columnas esperadas ({len(FEATURE_NAMES)}), en orden, con su tipo:")
for i, nombre in enumerate(FEATURE_NAMES, 1):
    tipo = "numérica" if nombre in cols_num else "categórica (texto)"
    print(f"  {i:2d}. {nombre}: {X_train[nombre].dtype} [{tipo}]")

# Verificación: recargar y predecir 3 filas
modelo_cargado = joblib.load(MODEL_OUTPUT_PATH)
muestra = X_test.head(3)
pred = np.asarray(modelo_cargado.predict(muestra))
prob = modelo_cargado.predict_proba(muestra)[:, 1]
esperado = (base_model.predict_proba(muestra)[:, 1] >= UMBRAL).astype(int)
print("Probabilidades:", np.round(prob, 4).tolist(), "| Predicciones:", pred.tolist(), "| Reales:", y_test.head(3).tolist())
assert pred.shape[0] == 3, "La verificación de exportación falló"
if UMBRAL_EMBEBIDO:
    assert pred.tolist() == esperado.tolist(), "predict() no coincide con el umbral evaluado"
print("Verificación de exportación OK.")